In [1]:
import numpy as np

# 1. 定義激發函數與交叉熵誤差函數
def relu(x):
    return np.maximum(0, x)

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def softmax(x):
    exp_x = np.exp(x - np.max(x)) # 防止溢位
    return exp_x / np.sum(exp_x)

def cross_entropy_error(y_hat, y):
    delta = 1e-7 # 避免 log(0)
    return -np.sum(y * np.log(y_hat + delta))

# 2. 準備輸入資料 x (6個特徵值)
x = np.array([1.2, 0.5, -0.8, 2.1, -1.0, 0.3])
print("輸入 x 形狀:", x.shape)

# 3. 初始化各層權重(W)與偏置(b)
# 第一隱藏層: 6 -> 5 (ReLU)
W1 = np.random.randn(6, 5)
b1 = np.random.randn(5)

# 第二隱藏層: 5 -> 4 (ReLU)
W2 = np.random.randn(5, 4)
b2 = np.random.randn(4)

# 第三隱藏層: 4 -> 4 (Sigmoid)
W3 = np.random.randn(4, 4)
b3 = np.random.randn(4)

# 輸出層: 4 -> 3 (Softmax)
W4 = np.random.randn(4, 3)
b4 = np.random.randn(3)

# 印出各層 W, b 的形狀與參數個數
print("\n--- 參數個數說明 ---")
print(f"W1形狀: {W1.shape}, b1形狀: {b1.shape} -> 參數個數: {W1.size + b1.size}")
print(f"W2形狀: {W2.shape}, b2形狀: {b2.shape} -> 參數個數: {W2.size + b2.size}")
print(f"W3形狀: {W3.shape}, b3形狀: {b3.shape} -> 參數個數: {W3.size + b3.size}")
print(f"W4形狀: {W4.shape}, b4形狀: {b4.shape} -> 參數個數: {W4.size + b4.size}")

# 4. 前向傳播計算
z1 = np.dot(x, W1) + b1
h1 = relu(z1)

z2 = np.dot(h1, W2) + b2
h2 = relu(z2)

z3 = np.dot(h2, W3) + b3
h3 = sigmoid(z3)

z4 = np.dot(h3, W4) + b4
y_hat = softmax(z4)

print("\n--- 各層計算結果與形狀 ---")
print("z1 形狀:", z1.shape, "| h1 (ReLU) 形狀:", h1.shape)
print("z2 形狀:", z2.shape, "| h2 (ReLU) 形狀:", h2.shape)
print("z3 形狀:", z3.shape, "| h3 (Sigmoid) 形狀:", h3.shape)
print("z4 形狀:", z4.shape, "| y_hat (Softmax) 結果:", y_hat, "| 形狀:", y_hat.shape)

# 5. 計算 Cross Entropy Error
y = np.array([0, 0, 1]) # 真實標籤為第 3 類
loss = cross_entropy_error(y_hat, y)
print(f"\n與真實標籤 y={y} 的 Cross Entropy Error: {loss:.4f}")


輸入 x 形狀: (6,)

--- 參數個數說明 ---
W1形狀: (6, 5), b1形狀: (5,) -> 參數個數: 35
W2形狀: (5, 4), b2形狀: (4,) -> 參數個數: 24
W3形狀: (4, 4), b3形狀: (4,) -> 參數個數: 20
W4形狀: (4, 3), b4形狀: (3,) -> 參數個數: 15

--- 各層計算結果與形狀 ---
z1 形狀: (5,) | h1 (ReLU) 形狀: (5,)
z2 形狀: (4,) | h2 (ReLU) 形狀: (4,)
z3 形狀: (4,) | h3 (Sigmoid) 形狀: (4,)
z4 形狀: (3,) | y_hat (Softmax) 結果: [0.01853449 0.00690219 0.97456332] | 形狀: (3,)

與真實標籤 y=[0 0 1] 的 Cross Entropy Error: 0.0258


In [2]:
# 1. 安裝並載入 gradio
!pip install gradio -q
import gradio as gr

# 2. 定義 dnn 函數 (使用第 1 題已設定好的 W1~W4, b1~b4)
def dnn(x_input):
    x_arr = np.array(x_input)

    # 逐層計算
    z1 = np.dot(x_arr, W1) + b1
    h1 = relu(z1)

    z2 = np.dot(h1, W2) + b2
    h2 = relu(z2)

    z3 = np.dot(h2, W3) + b3
    h3 = sigmoid(z3)

    z4 = np.dot(h3, W4) + b4
    y_hat_val = softmax(z4)

    return y_hat_val

# 比對步驟 1 的測試資料與 dnn() 輸出是否一致
y_hat_test = dnn(x)
print("dnn(x) 計算結果:", y_hat_test)
print("是否與第1題結果一致:", np.allclose(y_hat, y_hat_test))

# 3. 定義 Gradio 互動介面的預測函數
def predict_class(x1, x2, x3, x4, x5, x6):
    input_vector = [x1, x2, x3, x4, x5, x6]
    probabilities = dnn(input_vector)
    # 取得機率最高者的索引 (+1 轉為 1, 2, 3 類別)
    predicted_class = int(np.argmax(probabilities) + 1)
    return f"預測類別：第 {predicted_class} 類 (機率: {probabilities[predicted_class-1]:.4f})"

# 4. 設計 Gradio 介面
inputs = [gr.Number(value=0.0, label=f"特徵 x{i+1}") for i in range(6)]
output = gr.Textbox(label="預測結果")

demo = gr.Interface(
    fn=predict_class,
    inputs=inputs,
    outputs=output,
    title="DNN 類別預測器 (6特徵 -> 3類別)",
    description="請輸入 6 個特徵值，模型將輸出機率最高的類別 (1, 2, 或 3)。"
)

# 啟動 Gradio 介面
demo.launch(share=True)



dnn(x) 計算結果: [0.01853449 0.00690219 0.97456332]
是否與第1題結果一致: True
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://de44dc7ee8a647b297.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
